# PV control parameters

Goal: inspect the PV ramp tests and decide what the EMS can safely assume from the panel.

For each distance test I calculate:

PV power = corrected_voltage * corrected_current

Then I compare measured PV power to three EMS needs:

1. supply the maximum scaled load, 100 mW
2. charge the battery at the measured battery charge power
3. charge the PEM at the measured PEM electrolysis power

The important part is the feasibility check. A missing voltage threshold means the measured PV test
did not reach that requirement.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This makes the notebook work both from the repo root and from its own folder.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not find the project root folder containing data/")
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)


## 1. Load all PV ramp tests


In [ ]:
pv_folder = PROJECT_ROOT / "data/PV_test/New_test"
pv_files = sorted(pv_folder.glob("PV_*cm_ramp.csv"))

print("PV files:")
for file in pv_files:
    print(" -", file.name)


## 2. Correct the PV sensor readings

The PV data is measured on INA3. I use the same simple calibration used in the original analysis:

corrected_voltage = ina3_bus_V - 0.180

corrected_current_A = ina3_current_mA / 1000 + 0.000138

corrected_power_mW = corrected_voltage * corrected_current_A * 1000


In [ ]:
def load_pv_file(file):
    df = pd.read_csv(file)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["time_s"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
    df["pv_voltage_V"] = df["ina3_bus_V"] - 0.180
    df["pv_current_A"] = df["ina3_current_mA"] / 1000 + 0.000138
    df["pv_power_mW"] = df["pv_voltage_V"] * df["pv_current_A"] * 1000
    return df

pv_tests = {}
for file in pv_files:
    distance_cm = int(file.stem.split("_")[1].replace("cm", ""))
    pv_tests[distance_cm] = load_pv_file(file)

display(pv_tests[1].head())


## 3. Plot voltage, current, and power for each ramp


In [ ]:
plt.figure(figsize=(10, 4))
for distance_cm, df in pv_tests.items():
    plt.plot(df["time_s"], df["pv_power_mW"], label=f"{distance_cm} cm")
plt.title("PV power during load ramp")
plt.xlabel("Time [s]")
plt.ylabel("PV power [mW]")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(6, 5))
for distance_cm, df in pv_tests.items():
    plt.plot(df["pv_voltage_V"], df["pv_current_A"] * 1000, label=f"{distance_cm} cm")
plt.title("PV current-voltage curves")
plt.xlabel("PV voltage [V]")
plt.ylabel("PV current [mA]")
plt.legend()
plt.grid(True)
plt.show()


## 4. Define the EMS power requirements


In [ ]:
demand = pd.read_csv(PROJECT_ROOT / "app/python/data/variable_load_signal/scaled_may_power_profile_15min.csv")
battery_charge_summary = pd.read_csv(PROJECT_ROOT / "app/python/data/processed_Battery/battery_charge_summary.csv")
pem_charge_summary = pd.read_csv(PROJECT_ROOT / "app/python/data/processed_PEM/pem_analysis_charge_discharge_summary.csv")
pem_control = pd.read_csv(PROJECT_ROOT / "app/python/data/processed_PEM/pem_control_parameters.csv")

LOAD_REQUIREMENT_MW = demand["power_mW"].max()
BATTERY_CHARGE_REQUIREMENT_MW = battery_charge_summary["max_power_W"].iloc[0] * 1000

# Conservative PEM charging requirement: the largest measured average electrolysis power in the charge tests.
PEM_CHARGE_REQUIREMENT_MW = pem_charge_summary["avg_charge_power_W"].max() * 1000

# Lower theoretical/minimum electrolysis point, useful as a comparison but less conservative.
PEM_MINIMUM_ELECTROLYSIS_MW = pem_control["minimum_electrolysis_power_W"].iloc[0] * 1000

requirements = pd.DataFrame(
    {
        "requirement": [
            "maximum EMS load",
            "battery charging",
            "PEM charging, conservative measured case",
            "PEM minimum electrolysis point",
        ],
        "power_mW": [
            LOAD_REQUIREMENT_MW,
            BATTERY_CHARGE_REQUIREMENT_MW,
            PEM_CHARGE_REQUIREMENT_MW,
            PEM_MINIMUM_ELECTROLYSIS_MW,
        ],
    }
)
display(requirements)


## 5. Find MPP and first voltage that reaches each requirement

For a threshold voltage I use the first point in the ramp where PV power reaches the requirement.
This is intentionally conservative for the measured ramp order.


In [ ]:
def first_voltage_reaching_power(df, required_power_mW):
    reached = df[df["pv_power_mW"] >= required_power_mW]
    if reached.empty:
        return np.nan
    return reached.iloc[0]["pv_voltage_V"]

rows = []
for distance_cm, df in pv_tests.items():
    mpp_idx = df["pv_power_mW"].idxmax()
    mpp = df.loc[mpp_idx]
    rows.append(
        {
            "distance_cm": distance_cm,
            "mpp_time_s": mpp["time_s"],
            "mpp_voltage_V": mpp["pv_voltage_V"],
            "mpp_current_A": mpp["pv_current_A"],
            "mpp_power_mW": mpp["pv_power_mW"],
            "max_voltage_V": df["pv_voltage_V"].max(),
            "max_current_A": df["pv_current_A"].max(),
            "first_voltage_for_load_V": first_voltage_reaching_power(df, LOAD_REQUIREMENT_MW),
            "first_voltage_for_battery_charge_V": first_voltage_reaching_power(df, BATTERY_CHARGE_REQUIREMENT_MW),
            "first_voltage_for_pem_charge_V": first_voltage_reaching_power(df, PEM_CHARGE_REQUIREMENT_MW),
            "first_voltage_for_minimum_pem_electrolysis_V": first_voltage_reaching_power(df, PEM_MINIMUM_ELECTROLYSIS_MW),
        }
    )

pv_summary = pd.DataFrame(rows).sort_values("distance_cm")
display(pv_summary)


## 6. Critical feasibility check


In [ ]:
PV_MAX_POWER_MW = pv_summary["mpp_power_mW"].max()
PV_MAX_POWER_W = PV_MAX_POWER_MW / 1000

PV_MIN_LOAD_SUPPLY_VOLTAGE = pv_summary["first_voltage_for_load_V"].dropna().max()
PV_MIN_BATTERY_CHARGE_VOLTAGE = (
    pv_summary["first_voltage_for_battery_charge_V"].dropna().max()
    if pv_summary["first_voltage_for_battery_charge_V"].notna().any()
    else np.nan
)
PV_MIN_PEM_CHARGE_VOLTAGE = (
    pv_summary["first_voltage_for_pem_charge_V"].dropna().max()
    if pv_summary["first_voltage_for_pem_charge_V"].notna().any()
    else np.nan
)
PV_MIN_MINIMUM_ELECTROLYSIS_VOLTAGE = (
    pv_summary["first_voltage_for_minimum_pem_electrolysis_V"].dropna().max()
    if pv_summary["first_voltage_for_minimum_pem_electrolysis_V"].notna().any()
    else np.nan
)

print(f"PV max measured power: {PV_MAX_POWER_MW:.1f} mW")
print(f"Load requirement:      {LOAD_REQUIREMENT_MW:.1f} mW")
print(f"Battery charge need:   {BATTERY_CHARGE_REQUIREMENT_MW:.1f} mW")
print(f"PEM charge need:       {PEM_CHARGE_REQUIREMENT_MW:.1f} mW")
print(f"PEM minimum point:     {PEM_MINIMUM_ELECTROLYSIS_MW:.1f} mW")

print("\nCan PV supply maximum EMS load?", PV_MAX_POWER_MW >= LOAD_REQUIREMENT_MW)
print("Can PV charge the battery at measured charger power?", PV_MAX_POWER_MW >= BATTERY_CHARGE_REQUIREMENT_MW)
print("Can PV run conservative PEM charging?", PV_MAX_POWER_MW >= PEM_CHARGE_REQUIREMENT_MW)
print("Can PV reach minimum PEM electrolysis?", PV_MAX_POWER_MW >= PEM_MINIMUM_ELECTROLYSIS_MW)


## 7. Values to use in the app


In [ ]:
pv_parameters = pd.DataFrame(
    {
        "parameter": [
            "PV_MAX_POWER_W",
            "PV_MIN_LOAD_SUPPLY_VOLTAGE",
            "PV_MIN_BATTERY_CHARGE_VOLTAGE",
            "PV_MIN_PEM_CHARGE_VOLTAGE",
            "PV_MIN_MINIMUM_ELECTROLYSIS_VOLTAGE",
        ],
        "value": [
            PV_MAX_POWER_W,
            PV_MIN_LOAD_SUPPLY_VOLTAGE,
            PV_MIN_BATTERY_CHARGE_VOLTAGE,
            PV_MIN_PEM_CHARGE_VOLTAGE,
            PV_MIN_MINIMUM_ELECTROLYSIS_VOLTAGE,
        ],
        "unit": ["W", "V", "V", "V", "V"],
        "meaning": [
            "Maximum measured PV power",
            "Conservative voltage where measured PV can supply 100 mW load",
            "NaN means the measured PV did not reach battery charge power",
            "NaN means the measured PV did not reach conservative PEM charge power",
            "Voltage for the lower minimum electrolysis check, if reached",
        ],
    }
)

display(pv_parameters)
